In [1]:
# Cell 1: Setup and Imports

import pandas as pd
import numpy as np
import os
import torch
import torchvision.models as models
import torchvision.transforms as transforms
import lightgbm as lgb
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack
import warnings

# Import the function from your helper file
from my_utils import extract_image_features

warnings.filterwarnings('ignore')
print("All libraries imported successfully.")

All libraries imported successfully.


In [2]:
# Cell 2: Configuration and Data Loading

# --- Configuration ---
DATASET_DIRECTORY = 'dataset/'
IMAGE_DIRECTORY = os.path.join(DATASET_DIRECTORY, 'images')
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMAGE_BATCH_SIZE = 128
NUM_WORKERS = 6  # Set to the number of CPU cores you want to use for data loading
TEXT_MAX_FEATURES = 20000

print(f"Using device: {DEVICE}")

# --- Load DataFrames ---
train_dataframe = pd.read_csv(os.path.join(DATASET_DIRECTORY, 'train.csv'))
test_dataframe = pd.read_csv(os.path.join(DATASET_DIRECTORY, 'test.csv'))

# --- Prepare Image Paths ---
def get_image_path(image_url):
    image_filename = image_url.split('/')[-1]
    return os.path.join(IMAGE_DIRECTORY, image_filename)

train_dataframe['image_path'] = train_dataframe['image_link'].apply(get_image_path)
test_dataframe['image_path'] = test_dataframe['image_link'].apply(get_image_path)

# --- Clean Text Data ---
train_dataframe['catalog_content'].fillna('', inplace=True)
test_dataframe['catalog_content'].fillna('', inplace=True)

print("Data loaded and prepared.")
train_dataframe.head()

Using device: cuda
Data loaded and prepared.


,sample_id,catalog_content,image_link,price,image_path
0,33127,"Item Name: La Victoria Green Taco Sauce Mild, ...",https://m.media-amazon.com/images/I/51mo8htwTH...,4.89,dataset/images\51mo8htwTHL.jpg
1,198967,"Item Name: Salerno Cookies, The Original Butte...",https://m.media-amazon.com/images/I/71YtriIHAA...,13.12,dataset/images\71YtriIHAAL.jpg
2,261251,"Item Name: Bear Creek Hearty Soup Bowl, Creamy...",https://m.media-amazon.com/images/I/51+PFEe-w-...,1.97,dataset/images\51+PFEe-w-L.jpg
3,55858,Item Name: Judee’s Blue Cheese Powder 11.25 oz...,https://m.media-amazon.com/images/I/41mu0HAToD...,30.34,dataset/images\41mu0HAToDL.jpg
4,292686,"Item Name: kedem Sherry Cooking Wine, 12.7 Oun...",https://m.media-amazon.com/images/I/41sA037+Qv...,66.49,dataset/images\41sA037+QvL.jpg


In [3]:
# Cell 3: Text Feature Extraction (TF-IDF)

print("Starting text feature extraction...")
text_feature_vectorizer = TfidfVectorizer(
    max_features=TEXT_MAX_FEATURES,
    stop_words='english',
    ngram_range=(1, 2)
)

train_text_features = text_feature_vectorizer.fit_transform(train_dataframe['catalog_content'])
test_text_features = text_feature_vectorizer.transform(test_dataframe['catalog_content'])

print("Text features created. Shape:", train_text_features.shape)

Starting text feature extraction...
Text features created. Shape: (75000, 20000)


In [4]:
# Cell 4: Image Feature Extraction (ResNet-50 on GPU)

# --- Define Image Transformations ---
image_transformations = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# --- Load Pre-trained Model ---
print("Loading pre-trained ResNet-50 model...")
image_feature_extractor = models.resnet50(pretrained=True)
image_feature_extractor.fc = torch.nn.Identity()  # Remove final classification layer
image_feature_extractor = image_feature_extractor.to(DEVICE) # Move model to GPU
print("Model moved to GPU.")

# --- Run Feature Extraction ---
# This calls the function in my_utils.py and will use num_workers > 0 safely
train_image_features = extract_image_features(
    train_dataframe['image_path'].tolist(), 
    image_feature_extractor, 
    DEVICE, 
    IMAGE_BATCH_SIZE, 
    NUM_WORKERS, 
    image_transformations
)
test_image_features = extract_image_features(
    test_dataframe['image_path'].tolist(), 
    image_feature_extractor, 
    DEVICE, 
    IMAGE_BATCH_SIZE, 
    NUM_WORKERS, 
    image_transformations
)

print("Image features created.")
print("Train image features shape:", train_image_features.shape)

Loading pre-trained ResNet-50 model...
Model moved to GPU.


Extracting image features: 100%|█████████████████████████████████████████████████████| 586/586 [22:50<00:00,  2.34s/it]


Image features created.
Train image features shape: (75000, 2048)


In [6]:
# Cell 5: Combine All Features

print("Combining text and image features...")
combined_train_features = hstack([train_text_features, train_image_features]).tocsr()
combined_test_features = hstack([test_text_features, test_image_features]).tocsr()

print("Features combined. Final train shape:", combined_train_features.shape)

Combining text and image features...
Features combined. Final train shape: (75000, 22048)


In [7]:
# Cell 6 (New Version): Model Training and Validation

from sklearn.model_selection import train_test_split

# --- Prepare Target Variable ---
target_prices_log = np.log1p(train_dataframe['price'])

# --- Split the Data into Training and Validation Sets ---
# We'll use 80% of the data for training and 20% for validation.
X_train, X_val, y_train, y_val = train_test_split(
    combined_train_features,
    target_prices_log,
    test_size=0.2, # 20% for validation
    random_state=42 # Ensures the split is the same every time
)

print(f"Training on {X_train.shape[0]} samples, validating on {X_val.shape[0]} samples.")

# --- Define LightGBM Parameters for GPU ---
lgbm_params = {
    'device': 'gpu',
    'objective': 'regression_l1',
    'metric': 'rmse',
    'n_estimators': 2000,
    'learning_rate': 0.05,
    'num_leaves': 31,
    'verbose': -1,
    'seed': 42
}

# --- Train the Model on the Training Portion ONLY ---
print("Training LightGBM model on GPU...")
lgbm_regressor = lgb.LGBMRegressor(**lgbm_params)
lgbm_regressor.fit(X_train, y_train)
print("Model training complete.")

# --- Predict on the Validation Set and Calculate SMAPE ---
print("Predicting on validation set to calculate score...")
val_preds_log = lgbm_regressor.predict(X_val)

# Important: Convert both predictions and true labels back from log scale before scoring
val_preds_final = np.expm1(val_preds_log)
y_val_final = np.expm1(y_val)

# --- Calculate SMAPE directly using the formula ---
# Numerator: |predicted_price - actual_price|
numerator = np.abs(val_preds_final - y_val_final)
# Denominator: (|actual_price| + |predicted_price|) / 2
denominator = (np.abs(y_val_final) + np.abs(val_preds_final)) / 2
# The full formula: mean of ratio, as a percentage
validation_score = np.mean(numerator / (denominator + 1e-8)) * 100

print("\n" + "="*40)
print(f"✅ VALIDATION SMAPE SCORE: {validation_score:.4f}%")
print("="*40)
print("(Lower is better)")

Training on 60000 samples, validating on 15000 samples.
Training LightGBM model on GPU...
Model training complete.
Predicting on validation set to calculate score...

✅ VALIDATION SMAPE SCORE: 52.5808%
(Lower is better)


In [8]:
# Optional cell to run before final prediction

print("Retraining model on the ENTIRE training dataset for best performance...")
lgbm_regressor.fit(combined_train_features, target_prices_log)
print("Final model is ready.")

Retraining model on the ENTIRE training dataset for best performance...
Final model is ready.


In [9]:
print("Generating predictions for the test set...")

# Predict on the combined test features
log_predictions_test = lgbm_regressor.predict(combined_test_features)

# Inverse transform the predictions to get the actual price
final_price_predictions_test = np.expm1(log_predictions_test)

# Ensure all predicted prices are positive
final_price_predictions_test[final_price_predictions_test < 0] = 0

# Create the submission dataframe
submission_dataframe = pd.DataFrame({
    'sample_id': test_dataframe['sample_id'],
    'price': final_price_predictions_test
})

# Save the submission file
submission_dataframe.to_csv('submission.csv', index=False)

print("\n🚀 Submission file 'submission.csv' created successfully!")
submission_dataframe.head()

Generating predictions for the test set...

🚀 Submission file 'submission.csv' created successfully!


,sample_id,price
0,100179,18.129207
1,245611,17.405895
2,146263,18.870888
3,95658,8.661587
4,36806,30.042160


In [12]:
import joblib

# Define a filename for your model
model_filename = 'lgbm_price_model.joblib'

# Save the trained model object (lgbm_regressor) to the file
joblib.dump(lgbm_regressor, model_filename)

print(f"Model successfully saved to '{model_filename}'.")

Model successfully saved to 'lgbm_price_model.joblib'.
